In [10]:
import os
import numpy as np
import pandas as pd

In [11]:
# Import data:
# run script from the data_input file
os.chdir('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/data_input')


In [12]:
cluster_reps = pd.read_csv("representatives.csv")

features = pd.read_csv("featureset.csv", parse_dates=['Date'],na_values=['NA'])
features = features.set_index('Date')

add_features = pd.read_csv("factor_relative_features.csv", parse_dates=['Date'],na_values=['NA'])
add_features = add_features.set_index('Date')


print(add_features)

            zrel_momentum_vs_value_12  relvol_momentum_vs_value_12  \
Date                                                                 
2007-05-21                   0.903263                     0.007903   
2007-05-28                   1.507127                     0.008974   
2007-06-04                   0.522944                     0.009001   
2007-06-11                  -0.835358                     0.009225   
2007-06-18                   0.637502                     0.008506   
...                               ...                          ...   
2025-03-03                   2.149018                     0.015960   
2025-03-10                  -0.614738                     0.016265   
2025-03-17                   1.538872                     0.017896   
2025-03-24                   0.366983                     0.017405   
2025-03-31                   2.433776                     0.026013   

            corr_momentum_value_12  volratio_momentum_value_12  \
Date                   

In [13]:
# Cluster Reps
col2_values = cluster_reps['Representative'].unique()

# Filter dataset by checking whether its index
data= features[features.columns.intersection(col2_values)]

data = data.ffill().bfill()
print(data)

            VolTermStructPC1  VolTermStructPC2  MichgnConcIndx  \
Date                                                             
2007-03-05        -33.663206          8.061237            91.3   
2007-03-12        -26.590916          9.236760            91.3   
2007-03-19        -27.816286          9.407570            91.3   
2007-03-26        -25.510835          9.096880            91.3   
2007-04-02        -27.546502          9.140546            88.4   
...                      ...               ...             ...   
2025-03-03         44.421245         -5.261720            64.7   
2025-03-10         51.117954         -3.610349            64.7   
2025-03-17         42.305038         -6.657027            64.7   
2025-03-24         38.142130         -7.595933            64.7   
2025-03-31         44.285942         -6.064842            57.0   

            RandPPP_Factor_Inst_TS_ST  BCMPEBLS.Index  PPP.ZA.Index_QS  \
Date                                                               

In [14]:
# --- Drop features with more than 30% missing values ---
threshold = 0.30
missing_ratio = data.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > threshold].index
data = data.drop(columns=cols_to_drop)

print(data)

            VolTermStructPC1  VolTermStructPC2  MichgnConcIndx  \
Date                                                             
2007-03-05        -33.663206          8.061237            91.3   
2007-03-12        -26.590916          9.236760            91.3   
2007-03-19        -27.816286          9.407570            91.3   
2007-03-26        -25.510835          9.096880            91.3   
2007-04-02        -27.546502          9.140546            88.4   
...                      ...               ...             ...   
2025-03-03         44.421245         -5.261720            64.7   
2025-03-10         51.117954         -3.610349            64.7   
2025-03-17         42.305038         -6.657027            64.7   
2025-03-24         38.142130         -7.595933            64.7   
2025-03-31         44.285942         -6.064842            57.0   

            RandPPP_Factor_Inst_TS_ST  BCMPEBLS.Index  PPP.ZA.Index_QS  \
Date                                                               

In [15]:
df = pd.concat([data,add_features], axis=1, join='inner')
df = df.dropna()
lagged_data = df.shift(4).reset_index()
lagged_data = lagged_data.set_index('Date')
lagged_data= lagged_data.dropna()

print(df)


            VolTermStructPC1  VolTermStructPC2  MichgnConcIndx  \
Date                                                             
2007-05-21        -25.971465          9.547910            87.1   
2007-05-28        -26.283643          9.787659            87.1   
2007-06-04        -26.283553          9.942312            88.3   
2007-06-11        -28.132090          9.679810            88.3   
2007-06-18        -27.003497         10.308379            88.3   
...                      ...               ...             ...   
2025-03-03         44.421245         -5.261720            64.7   
2025-03-10         51.117954         -3.610349            64.7   
2025-03-17         42.305038         -6.657027            64.7   
2025-03-24         38.142130         -7.595933            64.7   
2025-03-31         44.285942         -6.064842            57.0   

            RandPPP_Factor_Inst_TS_ST  BCMPEBLS.Index  PPP.ZA.Index_QS  \
Date                                                               

In [16]:
# Save lagged dataset for models
lagged_data.to_csv('lagged_data.csv',index=True)

In [17]:
from sklearn.preprocessing import StandardScaler
import pandas as pd


scaler = StandardScaler()

# Fit on features only
X_scaled = scaler.fit_transform(lagged_data)

# Convert back to DataFrame with the same columns + index
X_scaled = pd.DataFrame(X_scaled, 
                        columns=lagged_data.columns, 
                        index=lagged_data.index)


print(X_scaled)

            VolTermStructPC1  VolTermStructPC2  MichgnConcIndx  \
Date                                                             
2007-06-18         -3.924906          2.545985        0.625376   
2007-06-25         -3.942785          2.585398        0.625376   
2007-07-02         -3.942780          2.610823        0.717581   
2007-07-09         -4.048646          2.567668        0.717581   
2007-07-16         -3.984011          2.671003        0.717581   
...                      ...               ...             ...   
2025-03-03         -0.198170         -0.121247       -0.557918   
2025-03-10         -0.369184         -0.333724       -0.557918   
2025-03-17         -0.404834         -0.365437       -0.557918   
2025-03-24         -0.149532         -0.168294       -0.557918   
2025-03-31          0.106501          0.111341       -1.095780   

            RandPPP_Factor_Inst_TS_ST  BCMPEBLS.Index  PPP.ZA.Index_QS  \
Date                                                               

In [18]:
# Save dataset for models
X_scaled.to_csv('scaled_data.csv',index=True)